# 04: Does pretraining matter? A vs B vs C, on all 80 COCO classes

Evaluates the three YOLO11n models on the **test** split: all 5,000 COCO val2017 images, the
official COCO benchmark. None of the three was trained or tuned on these images.

| | Model | Trained by |
|---|---|---|
| **A** | YOLO11n pretrained: the one Kontur uses (`weights/yolo11n.pt`) | Ultralytics |
| **B** | YOLO11n **from scratch** (`training/trained/yolo11n_B_scratch_coco.pt`) | us |
| **C** | YOLO11n fine-tuned from A (`training/trained/yolo11n_C_finetune_coco.pt`) | us |

All three detect the same 80 classes. **Run `training/train_yolo11n_coco.ipynb` first.**

**Metric:** the official COCO box AP (pycocotools), standard settings (confidence 0.001, up to
300 boxes per image), the same metric as notebook 02. **Sanity check:** A should land close to
Ultralytics' published 0.395 AP for YOLO11n on this benchmark; if it does, the evaluation is
measuring correctly.

**Outputs:** `eval/results/scratch_vs_pretrained/`.

## Settings

In [ ]:
from pathlib import Path
import json, os, sys, time

HERE = Path.cwd()                      # eval/ (Jupyter runs a notebook from its own folder)
if not (HERE / "04_scratch_vs_pretrained.ipynb").exists() and (HERE / "eval").is_dir():
    HERE = HERE / "eval"
ROOT = HERE.parent
TRAINING = ROOT / "training"
DATA_DIR = Path(os.environ.get("KONTUR_DATA", TRAINING / "data")).resolve()   # same place as the training notebook
OUT = HERE / "results" / "scratch_vs_pretrained"
OUT.mkdir(parents=True, exist_ok=True)

IMGSZ = 640
DEVICE = 0                  # first GPU; "cpu" also works (slower)
PUBLISHED_AP_A = 0.395      # Ultralytics' published COCO val2017 mAP50-95 for YOLO11n
MODELS = {
    "A_pretrained": ROOT / "weights" / "yolo11n.pt",
    "B_scratch": TRAINING / "trained" / "yolo11n_B_scratch_coco.pt",
    "C_finetune": TRAINING / "trained" / "yolo11n_C_finetune_coco.pt",
}
(ROOT / "weights").mkdir(exist_ok=True)    # git keeps no empty folders; A downloads into it
for tag, p in MODELS.items():
    print(f"{tag:13s} {p}  {'(found)' if p.exists() else '(missing)'}")

## The test split, as chosen by the training notebook

In [ ]:
import shutil, time, urllib.request
from pathlib import Path

UA = {"User-Agent": "Mozilla/5.0 (coco-download)"}

def fetch(url, dest, tries=4, timeout=60):
    """Download url to dest unless it is already there. Retries, then explains."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_name(dest.name + ".part")
    err = None
    for k in range(tries):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=timeout) as r, \
                 open(tmp, "wb") as f:
                shutil.copyfileobj(r, f, 1 << 20)
            tmp.replace(dest)
            return dest
        except Exception as e:
            err = e
            time.sleep(2 * (k + 1))
    raise RuntimeError(f"could not download {url}: {err}")

from pycocotools.coco import COCO

ds = json.loads((TRAINING / "results" / "dataset.json").read_text())
CLASSES = ds["classes"]
cat_ids = {n: int(c) for n, c in ds["coco_category_ids"].items()}
test_ids = ds["image_ids"]["test"]
coco_te = COCO(str(DATA_DIR / "annotations" / "instances_val2017.json"))
IMG = DATA_DIR / "images" / "test"
missing = [i for i in test_ids if not (IMG / coco_te.imgs[i]["file_name"]).exists()]
for i in missing:                       # normally none: the training notebook downloaded them
    fetch(coco_te.imgs[i]["coco_url"], IMG / coco_te.imgs[i]["file_name"])
print(f"test split: {len(test_ids)} images, {len(CLASSES)} classes, {len(missing)} fetched now")

## One evaluation, the same for all three models

In [ ]:
from pycocotools.cocoeval import COCOeval
from tqdm.auto import tqdm
from ultralytics import YOLO

def evaluate(weights, tag):
    model = YOLO(str(weights))          # A: downloaded into weights/ if missing
    files = [IMG / coco_te.imgs[i]["file_name"] for i in test_ids]
    dets, t0 = [], time.perf_counter()
    for k in tqdm(range(0, len(files), 32), desc=f"testing {tag}", mininterval=10):
        chunk_ids = test_ids[k:k + 32]
        for i, r in zip(chunk_ids, model.predict([str(f) for f in files[k:k + 32]], conf=0.001, iou=0.7,
                                                 max_det=300, imgsz=IMGSZ, device=DEVICE, verbose=False)):
            for (x1, y1, x2, y2), s, c in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist(), r.boxes.cls.tolist()):
                cat = cat_ids.get(model.names[int(c)])
                if cat is not None:
                    dets.append({"image_id": i, "category_id": cat, "bbox": [x1, y1, x2 - x1, y2 - y1], "score": s})
    sec = (time.perf_counter() - t0) / len(files)
    (OUT / f"detections_{tag}.json").write_text(json.dumps(dets))
    if not dets:
        return {"model": tag, "AP": 0.0, "AP50": 0.0}, {}
    ev = COCOeval(coco_te, coco_te.loadRes(dets), "bbox")
    ev.params.imgIds = test_ids
    ev.params.catIds = sorted(cat_ids.values())
    ev.evaluate(); ev.accumulate(); ev.summarize()
    s = ev.stats
    out = {"model": tag, "AP": s[0], "AP50": s[1], "AP75": s[2], "AP_small": s[3], "AP_medium": s[4],
           "AP_large": s[5], "AR100": s[8], "s_per_image": sec,
           "parameters": sum(p.numel() for p in model.model.parameters())}
    prec = ev.eval["precision"]                        # [IoU, recall, class, area, maxDets]
    per_class = {}
    for k, cid in enumerate(sorted(cat_ids.values())):
        p = prec[:, :, k, 0, -1]
        per_class[coco_te.cats[cid]["name"]] = float(p[p > -1].mean()) if (p > -1).any() else 0.0
    out = {k: (round(v, 4) if isinstance(v, float) else v) for k, v in out.items()}
    (OUT / f"metrics_{tag}.json").write_text(json.dumps({**out, "per_class_AP": per_class}, indent=1))
    return out, per_class

results, per_class = {}, {}
for tag, weights in MODELS.items():
    if tag != "A_pretrained" and not weights.exists():
        print(f"skipping {tag}: {weights} not found (train it first)")
        continue
    results[tag], per_class[tag] = evaluate(weights, tag)
if "A_pretrained" in results:
    print(f"\nSanity check: A scored AP {results['A_pretrained']['AP']:.3f}; Ultralytics publishes {PUBLISHED_AP_A}.")

## Compare

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

table = pd.DataFrame(list(results.values())).set_index("model")
table.to_csv(OUT / "comparison.csv")
display(table.T)

pc = pd.DataFrame(per_class)                 # rows = 80 classes, columns = models
pc.index.name = "class"
pc.to_csv(OUT / "per_class_ap.csv")
order = pc.sort_values(pc.columns[0], ascending=False).index
ax = pc.loc[order].plot.bar(figsize=(22, 5), width=0.85)
ax.set_ylabel("AP (COCO, IoU .50:.95)"); ax.set_xlabel("")
ax.set_title("Test AP per class (sorted by the first model): pretrained vs from scratch vs fine-tuned")
plt.tight_layout(); plt.savefig(OUT / "ap_per_class.png", dpi=130); plt.show()

# How far B gets towards A, class by class
if {"A_pretrained", "B_scratch"} <= set(pc.columns):
    ax = pc.plot.scatter(x="A_pretrained", y="B_scratch", figsize=(5, 5))
    ax.plot([0, 1], [0, 1], ls="--", c="grey"); ax.set_xlim(0, 0.8); ax.set_ylim(0, 0.8)
    ax.set_title("Per class: from scratch (B) vs pretrained (A)")
    plt.tight_layout(); plt.savefig(OUT / "scratch_vs_pretrained_per_class.png", dpi=150); plt.show()

# learning curves from the training notebook: how fast each starting point learns
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for tag in ("B_scratch", "C_finetune"):
    f = TRAINING / "results" / f"{tag}_results.csv"
    if f.exists():
        d = pd.read_csv(f); d.columns = [c.strip() for c in d.columns]
        axs[0].plot(d["epoch"], d["metrics/mAP50-95(B)"], label=tag)
        axs[1].plot(d["epoch"], d["val/box_loss"], label=tag)
if "A_pretrained" in results:
    axs[0].axhline(results["A_pretrained"]["AP"], ls="--", c="grey", label="A pretrained (test AP)")
axs[0].set_title("val mAP50-95 per epoch"); axs[1].set_title("val box loss per epoch")
for a in axs: a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.savefig(OUT / "learning_curves.png", dpi=150); plt.show()

## How to read the result

- **A close to 0.395** confirms the evaluation measures the same way as Ultralytics.
- **B below A** is the expected and useful finding: the same network, trained by us from random
  weights for a fixed budget, learns far less than weights pretrained by Ultralytics for 600
  epochs. This is the evidence for Kontur's choice of pretrained models.
- **C vs A** shows whether a further fine-tune on COCO changes the pretrained model.
- The learning curves show how fast each starting point learns (val split; A's dashed line is
  its test AP, a rough guide only; the table is the comparison).